# Web Scraping Part-level Information

This section scrapes information for every aircraft part in the sales dataset from an online aviation parts database. It takes 2,748 unique part numbers, then searches each part number on aviationsourcingsolutions.com to pull back its ATA chapter, platform and component details.

In [ ]:
import pandas as pd

path = "Sale_data.xlsx"

# Load the excel file (first sheet by default)
df = pd.read_excel(path)
# Labels to remove
exclude_parts = [
    "TRANSPORT",
    "CORE REPAIR CHARGE",
    "LATE FEE",
    "Dangerous Goods (DG FEE)",
    "FEE"
]

exclude_set = {x.strip().upper() for x in exclude_parts}

parts_series = (
    df["Item: Part Number"]
    .astype(str)
    .str.strip()
    .replace({"nan": None, "None": None, "": None})
    .dropna()
)

# remove excluded labels (exact match, case-insensitive)
parts_series = parts_series[~parts_series.str.upper().isin(exclude_set)]

# dedupe while preserving order
parts_unique = list(dict.fromkeys(parts_series.tolist()))

# first 100
parts_first_100 = parts_unique[:100]

print("Unique parts:", len(parts_unique))
print("Scraping first 100:", len(parts_first_100))
print("First 10:", parts_first_100[:10])


Unique parts: 2748
Scraping first 100: 100
First 10: ['1856M15G02', '326975', '53H687-01', '12099', '338-006-202-0', '45332-8039', '15-0810-5', '820914-7', '1649M27G01', 'RD-FA1041-01']


In [ ]:
import re
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE = "https://www.aviationsourcingsolutions.com"
SEARCH_TMPL = BASE + "/partno-search?searchby=partno&searchkey={part}"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; parts-scraper/1.0)",
    "Accept-Language": "en-US,en;q=0.9",
}

session = requests.Session()
session.headers.update(HEADERS)

def _norm_pn(pn: str) -> str:
    return re.sub(r"[^A-Z0-9]", "", (pn or "").upper())

def _get_html(url: str) -> str:
    r = session.get(url, timeout=30)
    r.raise_for_status()
    return r.text

def _find_first_table_by_title(soup: BeautifulSoup, title_contains: str):
    for card in soup.select("div.card.custom_card"):
        h3 = card.select_one("h3.table-title")
        title = h3.get_text(" ", strip=True) if h3 else ""
        if title_contains in title:
            table = card.select_one("table")
            if table:
                return table
            return None
    return None

def _extract_matching_rows_keyvaluepairs(
    table,
    input_part_number: str,
    part_col_index: int = 0,
    add_part_link: bool = True
) -> list[str]:

    headers = [th.get_text(" ", strip=True) for th in table.select("thead th")]
    target = _norm_pn(input_part_number)

    kvps = []
    for tr in table.select("tbody tr"):
        tds = tr.find_all("td", recursive=False)
        if not tds:
            continue

        # Build header to value dict
        row_dict = {}
        for i, td in enumerate(tds):
            key = headers[i] if i < len(headers) else f"col_{i}"
            row_dict[key] = td.get_text(" ", strip=True)

        # Determine part value from first column (default)
        part_header = headers[part_col_index] if headers else "Part No."
        part_value = row_dict.get(part_header, tds[part_col_index].get_text(" ", strip=True))

        # Only keep exact matches to the input PN
        if _norm_pn(part_value) != target:
            continue

        # Add link from the part cell if present (useful info, but we won't follow it)
        if add_part_link:
            a = tds[part_col_index].find("a", href=True)
            if a:
                row_dict[f"{part_header} URL"] = urljoin(BASE, a["href"])

        # Make KeyValuePairs (single field)
        kvp = " | ".join([f"{k}: {v}" for k, v in row_dict.items() if v not in ("", None)])
        kvps.append(kvp)

    return kvps

def scrape_one_part_keyvalue(part_number: str, sleep_s: float = 1.2) -> dict:
    out = {
        "_input_part_number": part_number,
        "_status": "",
        "_error": "",
        "KeyValuePairs": ""
    }

    try:
        html = _get_html(SEARCH_TMPL.format(part=part_number))
        soup = BeautifulSoup(html, "html.parser")

        # Try ATA table first
        ata_table = _find_first_table_by_title(soup, "Search Result For ATA Part Number")
        if ata_table is not None:
            ata_kvps = _extract_matching_rows_keyvaluepairs(ata_table, part_number, part_col_index=0, add_part_link=True)
            if ata_kvps:
                # If multiple matching rows, join them clearly
                out["_status"] = "ok_ata"
                out["KeyValuePairs"] = "\n---\n".join(ata_kvps)
                return out
            else:
                # ATA table exists but no matching PN row
                out["_status"] = "no_match_in_ata"
                out["KeyValuePairs"] = ""
                return out

        # Fallback to Aviation Component table if ATA table is missing
        comp_table = _find_first_table_by_title(soup, "Search Result For Aviation Component")
        if comp_table is not None:
            comp_kvps = _extract_matching_rows_keyvaluepairs(comp_table, part_number, part_col_index=0, add_part_link=True)
            if comp_kvps:
                out["_status"] = "ok_component_fallback"
                out["KeyValuePairs"] = "\n---\n".join(comp_kvps)
                return out
            else:
                out["_status"] = "no_match_in_component"
                out["KeyValuePairs"] = ""
                return out

        # Neither table exists
        out["_status"] = "no_table_ata_or_component"
        return out

    except Exception as e:
        out["_status"] = "error"
        out["_error"] = repr(e)
        return out

    finally:
        time.sleep(sleep_s)

def scrape_all_to_csv(
    parts: list[str],
    sleep_s: float = 1.2,
    out_csv: str = "all_parts.csv",
    checkpoint_every: int = 50
):
    rows = []
    total = len(parts)

    for i, pn in enumerate(parts, 1):  # <-- change: iterate over ALL parts
        rows.append(scrape_one_part_keyvalue(pn, sleep_s=sleep_s))

        # taking time between scrapes
        if i % 25 == 0:
            time.sleep(max(2.0, sleep_s))

        # checkpoint save to not lose progress
        if checkpoint_every and (i % checkpoint_every == 0):
            pd.DataFrame(rows, columns=["_input_part_number", "_status", "_error", "KeyValuePairs"])\
              .to_csv(out_csv, index=False)
            print(f"Checkpoint saved: {i}/{total}")

    df = pd.DataFrame(rows, columns=["_input_part_number", "_status", "_error", "KeyValuePairs"])
    df.to_csv(out_csv, index=False)
    print(f"Done. Saved: {out_csv}")
    return df

# Run it:
#df_out = scrape_all_to_csv(parts_unique, sleep_s=0.2, out_csv="ATA_chapters_all_parts.csv", checkpoint_every=50)



Checkpoint saved: 50/2748
Checkpoint saved: 100/2748
Checkpoint saved: 150/2748
Checkpoint saved: 200/2748
Checkpoint saved: 250/2748
Checkpoint saved: 300/2748
Checkpoint saved: 350/2748
Checkpoint saved: 400/2748
Checkpoint saved: 450/2748
Checkpoint saved: 500/2748
Checkpoint saved: 550/2748
Checkpoint saved: 600/2748
Checkpoint saved: 650/2748
Checkpoint saved: 700/2748
Checkpoint saved: 750/2748
Checkpoint saved: 800/2748
Checkpoint saved: 850/2748
Checkpoint saved: 900/2748
Checkpoint saved: 950/2748
Checkpoint saved: 1000/2748
Checkpoint saved: 1050/2748
Checkpoint saved: 1100/2748
Checkpoint saved: 1150/2748
Checkpoint saved: 1200/2748
Checkpoint saved: 1250/2748
Checkpoint saved: 1300/2748
Checkpoint saved: 1350/2748
Checkpoint saved: 1400/2748
Checkpoint saved: 1450/2748
Checkpoint saved: 1500/2748
Checkpoint saved: 1550/2748
Checkpoint saved: 1600/2748
Checkpoint saved: 1650/2748
Checkpoint saved: 1700/2748
Checkpoint saved: 1750/2748
Checkpoint saved: 1800/2748
Checkpoint s